In [ ]:
GPU="1"
num_GPUs = 1
gen_image_batch_size=128

In [ ]:
import os
os.environ['PATH'] += ':/home/temp0/anaconda3/envs/edm2/bin'

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

def show_images_from_dir(folder_path, num_images=4, start_seed=42):
    """
    顯示指定資料夾中的圖片（根據檔名排序，從指定種子碼開始），以 2x2 格顯示。

    Args:
        folder_path (str): 圖片所在資料夾路徑
        num_images (int): 要顯示的圖片數量（預設 4）
        start_seed (int): 從第幾個種子（圖片）開始（根據檔名排序）
    """
    if not os.path.isdir(folder_path):
        print(f"[錯誤] 資料夾不存在：{folder_path}")
        return

    # 過濾圖片檔案，並根據檔名中的數字排序
    image_files = sorted([
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ], key=lambda x: int(''.join(filter(str.isdigit, os.path.basename(x))) or 0))

    # 找到起始 index
    start_index = next((i for i, f in enumerate(image_files)
                        if int(''.join(filter(str.isdigit, os.path.basename(f))) or 0) >= start_seed), None)

    if start_index is None:
        print(f"[警告] 找不到 seed >= {start_seed} 的圖片。")
        return

    subset = image_files[start_index:start_index + num_images]
    if not subset:
        print(f"[警告] 從 seed {start_seed} 起沒有足夠圖片可顯示。")
        return

    # 顯示 2x2 圖片
    rows = cols = 2
    plt.figure(figsize=(8, 8))
    for i, img_path in enumerate(subset):
        img = Image.open(img_path)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# 特殊條件v6 restricted_w scheduler,noise gating


In [ ]:

w=2.1
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.3
hold_steps = 3
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=False \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=1.7
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 100
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.1
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 100
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# 測試永遠開啟是否正確對應CFG constant

In [ ]:

w=1.7
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.0
close_prob = -1.0
hold_steps = 3
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_const_testing_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=False \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=1.7
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 100
adaptive_prob = 0.5
open_prob = 0.0
close_prob = -1.0
hold_steps = 3
dirname=f'/data/guidance-team/out/2025_0827_const_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)

# 測試永遠開啟但w interval是否正確對應CFG interval

In [ ]:
w=2.1
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 100
adaptive_prob = 0.5
open_prob = 0.0
close_prob = -1.0
hold_steps = 3
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_interval_testing3_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=interval_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=False \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.1
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 100
adaptive_prob = 0.5
open_prob = 0.0
close_prob = -1.0
hold_steps = 3
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_interval_testing2_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/generate_images_for_test.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg \
--guidance_scheduler=interval_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=False \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# 正式大量測試 V6

# restricted_linear_increase_scheduler

In [ ]:

w=2.1
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=False \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=1.7
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=False \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.1
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.3
hold_steps = 2
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=False \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.1
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=False \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.1
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.5
hold_steps = 1
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=False \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=1.7
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.5
hold_steps = 1
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=False \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.1
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 2
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.5
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 2
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# 同上 但使用restricted skewed sin schedule

In [ ]:

w=2.5
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 2
w_skewed = 0.38
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_skewed_sinusoidal_schedule \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:

w=2.5
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 2
w_skewed = 0.38
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v6_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=mid_activate_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# V7 Using scheduler open prob, close prob

In [ ]:

w=2.5
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.3
hold_steps = 2
w_skewed = 0.38
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# V7 Using scheduler open prob, close prob 提高prob

In [ ]:

w=2.5
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.5
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 3.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.5
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.7
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 3.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.5
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.7
close_prob = 0.5
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.5
w_interval_low_steps = 10
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.5
w_interval_low_steps = 15
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=3.0
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=3.5
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=4.0
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.5
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.0
w_interval_low_steps = 5
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.0
w_interval_low_steps = 15
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.5
w_interval_low_steps = 15
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.7
close_prob = 0.5
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.5
w_interval_low_steps = 15
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.0
w_interval_low_steps = 15
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=3.0
w_interval_low_steps = 15
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.75
w_interval_low_steps = 15
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v7_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive \
--guidance_scheduler=restricted_linear_increase_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# V8 General by diff of true prob

In [ ]:
w=2.5
w_interval_low_steps = 15
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 10.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v8_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_general \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.5
w_interval_low_steps = 15
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.6
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 8.0
dirname=f'/data/guidance-team/out/2025_0827_Adaptive_v8_mid_gating_CFG_v9_w={w},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=cfg_adaptive_general \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

# MAP+V7Adaptive

In [ ]:
w=2.5
w_interval_low_steps = 15
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
n = 1
k = 1
dirname=f'/data/guidance-team/out/2025_0827_MAP_Adaptive_v8_mid_gating_CFG_v9_w={w},n={n},k={k},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=map_adaptive \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--n_random={n} \
--k_average={k} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.0
w_interval_low_steps = 15
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
n = 1
k = 1
dirname=f'/data/guidance-team/out/2025_0827_MAP_Adaptive_v8_mid_gating_CFG_v9_w={w},n={n},k={k},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=map_adaptive \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--n_random={n} \
--k_average={k} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
w=2.0
w_interval_low_steps = 15
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
n = 1
k = 1
dirname=f'/data/guidance-team/out/2025_0827_MAP_Adaptive_v8_mid_gating_CFG_v9_w={w},n={n},k={k},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'

show_images_from_dir(f"/{dirname}/000000", start_seed=4)

In [ ]:
w=1.5
w_interval_low_steps = 15
w_interval_high_steps = 30
num_images = 10000
adaptive_prob = 0.5
open_prob = 0.5
close_prob = 0.4
hold_steps = 2
w_skewed = 0.38
balance_coeff = 4.0
n = 1
k = 1
dirname=f'/data/guidance-team/out/2025_0827_MAP_Adaptive_v8_mid_gating_CFG_v9_w={w},n={n},k={k},{w_interval_low_steps},{w_interval_high_steps},open_prob={open_prob},close_prob={close_prob},hold_steps={hold_steps}'
# restricted_linear_increase_scheduler
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking.py \
--preset=edm2-img512-s-guid-fid \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images+10} \
--sampler=map_adaptive \
--guidance_scheduler=const_scheduler \
--w_interval_low_steps={w_interval_low_steps} \
--w_interval_high_steps={w_interval_high_steps} \
--w_skewed={w_skewed} \
--adaptive_prob={adaptive_prob} \
--open_prob={open_prob} \
--close_prob={close_prob} \
--hold_steps={hold_steps} \
--balance_coeff={balance_coeff} \
--n_random={n} \
--k_average={k} \
--debug=False \
--guidance={w} \
--heun_guid=True \
--batch={gen_image_batch_size} \
--save_classifier_stats=True \
--classifier_path=/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt \
--image_size 64 \
--classifier_attention_resolutions 32,16,8 \
--classifier_depth 4 \
--classifier_width 128 \
--classifier_pool attention \
--classifier_resblock_updown True \
--classifier_use_scale_shift_norm True
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000